In [1]:
import sys
try:
    import google.colab

    from google.colab import drive
    drive.mount('/content/drive')
    !git clone https://github.com/max15189/InverseKinematics.git

    sys.path.insert(0, '/content/InverseKinematics')
    IN_COLAB = True

except ImportError:
    IN_COLAB = False

    sys.path.insert(0, r"c:\Users\max\InverseKinematics")

SAVE_DATA = "/content/drive/MyDrive/inverse_kinematics/pos_rot_qinit" if IN_COLAB else "G:/My Drive/inverse_kinematics/pos_rot_qinit"
SAVE_MODEL = f"{SAVE_DATA}/mlp_ik.pt"

BATCH_SIZE = 512



import torch



DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")



device: cpu


In [3]:
from torch.utils.data import DataLoader

from ik.data.dataset import IKDataset
train_ds = IKDataset("train", SAVE_DATA)
val_ds   = IKDataset("val",   SAVE_DATA,
                     MinMax_X=train_ds.MinMax_X,
                     MinMax_Y=train_ds.MinMax_Y)

test_ds  = IKDataset("test",  SAVE_DATA,
                     MinMax_X=train_ds.MinMax_X,
                     MinMax_Y=train_ds.MinMax_Y)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"train: {len(train_ds):,} samples")
print(f"val:   {len(val_ds):,} samples")
print(f"test:  {len(test_ds):,} samples")
print(f"input dim: {train_ds.X.shape[1]}  (R6[6] + P[3] + q_init[6])")
print(f"output dim: {train_ds.y.shape[1]}  (q_target[6])")


train: 240,000 samples
val:   30,000 samples
test:  30,000 samples
input dim: 15  (R6[6] + P[3] + q_init[6])
output dim: 6  (q_target[6])


In [4]:
from ik.model.mlp import MLP

model = MLP(input_dim=15, hidden_dim=256, output_dim=6, n_layers=4).to(DEVICE)
model.load_state_dict(torch.load(SAVE_MODEL))
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
print(model)

parameters: 203,014
MLP(
  (net): Sequential(
    (0): Linear(in_features=15, out_features=256, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): LeakyReLU(negative_slope=0.01)
    (4): Linear(in_features=256, out_features=256, bias=True)
    (5): LeakyReLU(negative_slope=0.01)
    (6): Linear(in_features=256, out_features=256, bias=True)
    (7): LeakyReLU(negative_slope=0.01)
    (8): Linear(in_features=256, out_features=6, bias=True)
  )
)


In [5]:
from ik.model.training import _denorm_q,_to_tensors
test_ds=IKDataset("test",SAVE_DATA,train_ds.MinMax_X,train_ds.MinMax_Y)
Y_min,Y_max=_to_tensors(test_ds.MinMax_Y,DEVICE)
q_predict=model(torch.tensor(test_ds.X))
q_predict=_denorm_q(q_predict,Y_min,Y_max)
q_predict=q_predict.cpu().detach().numpy()

In [ ]:
if IN_COLAB:
     ! pip install modern-robotics
     ! pip install tdqm
import numpy as np
import modern_robotics as mr
def IKinSpace_with_iters(Slist, M, T, thetalist0, eomg, ev):
    thetalist = np.array(thetalist0).copy()
    i = 0
    maxiterations = 50
    Tsb = mr.FKinSpace(M,Slist, thetalist)
    Vs = np.dot(mr.Adjoint(Tsb),mr.se3ToVec(mr.MatrixLog6(np.dot(mr.TransInv(Tsb), T))))
    err = np.linalg.norm([Vs[0], Vs[1], Vs[2]]) > eomg or np.linalg.norm([Vs[3], Vs[4], Vs[5]]) >  ev
    while err and i < maxiterations:
        thetalist = thetalist + np.dot(np.linalg.pinv(mr.JacobianSpace(Slist,thetalist)), Vs)
        i = i + 1
        Tsb = mr.FKinSpace(M, Slist, thetalist)
        Vs = np.dot(mr.Adjoint(Tsb), \
                    mr.se3ToVec(mr.MatrixLog6(np.dot(mr.TransInv(Tsb), T))))
        err = np.linalg.norm([Vs[0], Vs[1], Vs[2]]) > eomg \
              or np.linalg.norm([Vs[3], Vs[4], Vs[5]]) > ev
    return (thetalist, not err,i)

In [8]:
import numpy as np
from ik.kinematics.fk import FK,_Slist_np,_M_HOME_np
from tqdm import tqdm
iterations_list = []
convergence_list = []

for i in tqdm(range(len(q_predict))): # Iterate through all predicted configurations
    # Use the true target pose for the IKinSpace_with_iters function
    target_T = FK(test_ds.q_target[i])
    initial_q = q_predict[i]

    # Call IKinSpace_with_iters
    thetalist_out, converged, num_iters = IKinSpace_with_iters(_Slist_np, _M_HOME_np, target_T, initial_q, 0.001, 0.001)

    iterations_list.append(num_iters)
    convergence_list.append(converged)


average_iterations = np.mean(iterations_list)
percentage_convergence = np.sum(convergence_list) / len(convergence_list) * 100

print(f"Average iterations: {average_iterations:.2f}")
print(f"Percentage of convergence: {percentage_convergence:.2f}%")

100%|██████████| 30000/30000 [02:24<00:00, 207.06it/s]

Average iterations: 1.50
Percentage of convergence: 100.00%


## Sample configs from joint limits and compute FK

In [10]:
from ik.kinematics.fk import FK, JOINT_LIMITS
import numpy as np

def sample_fk(n, seed=42):

    T_samples = np.array([FK(q) for q in q_samples])  # (n, 4, 4)

    return T_samples, q_samples

T_samples, q_samples = sample_fk(1000)
print(T_samples.shape, q_samples.shape)
type(T_samples)

(1000, 4, 4) (1000, 6)


numpy.ndarray

In [ ]:
np.random.seed(42)
q_samples = np.random.uniform(
    JOINT_LIMITS[:, 0], JOINT_LIMITS[:, 1], size=(1000, 6)
    
)

In [ ]:
q_predict=model(torch.tensor(test_ds.X))
q_predict=_denorm_q(q_predict,Y_min,Y_max)
q_predict=q_predict.cpu().detach().numpy()

In [14]:
import numpy as np
from ik.kinematics.fk import FK,_Slist_np,_M_HOME_np
from tqdm import tqdm
iterations_list = []
convergence_list = []

for i in tqdm(range(len(q_predict))): # Iterate through all predicted configurations
    # Use the true target pose for the IKinSpace_with_iters function
    target_T = FK(test_ds.q_target[i])
    initial_q = q_predict[i]

    # Call IKinSpace_with_iters
    thetalist_out, converged, num_iters = IKinSpace_with_iters(_Slist_np, _M_HOME_np, T_samples[i], initial_q, 0.001, 0.001)

    iterations_list.append(num_iters)
    convergence_list.append(converged)


average_iterations = np.mean(iterations_list)
percentage_convergence = np.sum(convergence_list) / len(convergence_list) * 100

print(f"Average iterations: {average_iterations:.2f}")
print(f"Percentage of convergence: {percentage_convergence:.2f}%")

 32%|███▏      | 323/1000 [00:09<00:19, 35.22it/s]


KeyboardInterrupt: 

In [15]:
def make_X(T_targets, q_current, MinMax_X):
    """
    T_targets  : (n, 4, 4)
    q_current  : (n, 6)
    MinMax_X   : [min (15,), max (15,)] from train_ds.MinMax_X
    returns X  : (n, 15) normalised to [-1, 1]
    """
    R6 = T_targets[:, :2, :3].reshape(len(T_targets), 6)  # (n, 6)
    P  = T_targets[:, :3, 3]                               # (n, 3)
    X  = np.concatenate([R6, P, q_current], axis=1)        # (n, 15)
    X  = (X - MinMax_X[0]) / (MinMax_X[1] - MinMax_X[0])
    X  = (X * 2) - 1
    return X.astype(np.float32)

def sample_random_X(n, MinMax_X, seed=42):
    """
    Samples n random (T_target, q_current) pairs from joint space.
    Returns X (n, 15) normalised, T_targets (n, 4, 4), q_targets (n, 6), q_currents (n, 6).
    """
    np.random.seed(seed)
    q_targets  = np.random.uniform(JOINT_LIMITS[:, 0], JOINT_LIMITS[:, 1], size=(n, 6))
    q_currents = np.random.uniform(JOINT_LIMITS[:, 0], JOINT_LIMITS[:, 1], size=(n, 6))
    T_targets  = np.array([FK(q) for q in q_targets])     # (n, 4, 4)
    X          = make_X(T_targets, q_currents, MinMax_X)
    return X, T_targets, q_targets, q_currents


In [25]:
X, T_targets, q_targets, q_currents = sample_random_X(1000, train_ds.MinMax_X)

In [26]:
q_predict=model(torch.tensor(X))
q_predict=_denorm_q(q_predict,Y_min,Y_max)
q_predict=q_predict.cpu().detach().numpy()

In [30]:
import numpy as np
from ik.kinematics.fk import FK,_Slist_np,_M_HOME_np
from tqdm import tqdm
iterations_list = []
convergence_list = []

for i in tqdm(range(len(q_predict))): # Iterate through all predicted configurations
    # Use the true target pose for the IKinSpace_with_iters function
    target_T = T_targets[i,:,:]
    initial_q = q_predict[i]

    # Call IKinSpace_with_iters
    thetalist_out, converged, num_iters = IKinSpace_with_iters(_Slist_np, _M_HOME_np, target_T, q_currents[i], 0.001, 0.001)

    iterations_list.append(num_iters)
    convergence_list.append(converged)


average_iterations = np.mean(iterations_list)
percentage_convergence = np.sum(convergence_list) / len(convergence_list) * 100

print(f"Average iterations: {average_iterations:.2f}")
print(f"Percentage of convergence: {percentage_convergence:.2f}%")

100%|██████████| 1000/1000 [00:27<00:00, 36.18it/s]

Average iterations: 14.93
Percentage of convergence: 98.20%


In [31]:
import numpy as np
from ik.kinematics.fk import FK,_Slist_np,_M_HOME_np
from tqdm import tqdm
iterations_list = []
convergence_list = []

for i in tqdm(range(len(q_predict))): # Iterate through all predicted configurations
    # Use the true target pose for the IKinSpace_with_iters function
    target_T = T_targets[i,:,:]
    initial_q = q_predict[i]

    # Call IKinSpace_with_iters
    thetalist_out, converged, num_iters = IKinSpace_with_iters(_Slist_np, _M_HOME_np, target_T, initial_q, 0.001, 0.001)

    iterations_list.append(num_iters)
    convergence_list.append(converged)


average_iterations = np.mean(iterations_list)
percentage_convergence = np.sum(convergence_list) / len(convergence_list) * 100

print(f"Average iterations: {average_iterations:.2f}")
print(f"Percentage of convergence: {percentage_convergence:.2f}%")

100%|██████████| 1000/1000 [00:18<00:00, 52.80it/s]

Average iterations: 9.77
Percentage of convergence: 99.10%
